# 059 — Transferencia, fine-tuning y destilación

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución explicada

**Ejercicio 1.** Full: 1024² = 1 048 576. LoRA: r·(1024+1024) = 2048·r →
r=4: **8192 (0.78 %)**; r=16: **32 768 (3.13 %)**; r=64: **131 072 (12.5 %)**.
El coste crece lineal en r; la práctica común usa r entre 4 y 64 según la distancia
entre tarea y preentrenamiento.

**Ejercicio 2.** T=1: (e⁴, e², e⁰)/Σ = (54.60, 7.39, 1)/62.99 =
**(0.867, 0.117, 0.016)**. T=2: z/T = (2, 1, 0) → (7.389, 2.718, 1)/11.107 =
**(0.665, 0.245, 0.090)**. Con T=2 el student ve que la clase 2 es 2.7× más plausible
que la 3 (con T=1 esa relación queda comprimida cerca de cero): esa estructura
relativa es el "conocimiento oscuro" que se destila.

**Ejercicio 3.** Con 300 imágenes y dominio cercano: **feature extraction** (o LoRA
con r pequeño). El fine-tuning completo con 300 ejemplos sobreajusta y arriesga
olvido; el tronco preentrenado ya trae las features adecuadas para un dominio similar.

**Ejercicio 4.** Evaluar el modelo *antes y después* del fine-tuning sobre: (a) un
benchmark general del dominio original (p. ej. un subconjunto de validación de
ImageNet o tareas estándar del LLM), y (b) la tarea nueva. Olvido catastrófico =
mejora en (b) con caída significativa en (a); mitigaciones: η menor, menos épocas,
congelar más capas o PEFT.


In [ ]:
result = run_lab("neural", seed=59)
assert result["kind"] == "neural"
assert result["evidence"]
show(result)


In [ ]:
# Verificación numérica
import math

# Ejercicio 1
d = k = 1024
full = d * k
for r in (4, 16, 64):
    params = r * (d + k)
    print(f"r={r}: {params} params ({100*params/full:.2f} %)")
assert 4 * (d + k) == 8192

# Ejercicio 2
def softmax_T(z, T):
    e = [math.exp(v / T) for v in z]
    s = sum(e)
    return [round(v / s, 3) for v in e]

z = [4.0, 2.0, 0.0]
print("T=1:", softmax_T(z, 1), "| T=2:", softmax_T(z, 2))


## Reflexión

1. ¿Por qué la corrección ΔW aprendida al adaptar una tarea suele tener rango bajo, y qué pasaría con LoRA si no lo tuviera?
2. ¿Qué información contiene la distribución suavizada del teacher que no contiene la etiqueta dura, y por qué T=1 la oculta?
3. Antes de hacer fine-tuning de un modelo base ajeno, ¿qué deberías medir de ese modelo además de su accuracy en tu tarea?
